# NLP Tutorial 1 — French Environmental Corpus (Group 3, French)

**Project:** *French Environmental Text Intelligence — automatic classification and information retrieval in French environmental texts.*

This week builds the corpus that the project will reuse later (tokenization, n-grams, TF-IDF, classification, search).

| | |
|---|---|
| Language | French (`fr_core_news_md`) |
| Text type | Introductions of French Wikipedia articles |
| Categories | émissions · énergies renouvelables · pollution · politique climatique |
| Licence | CC BY-SA 4.0 |

Run with **Restart and run all** before saving (section 6.7).

## 1. Setup — spaCy and the French model

In [1]:
# Uncomment on Google Colab or on a new machine:
# !pip install -U spacy
# !python -m spacy download fr_core_news_md

In [2]:
import spacy
from spacy import displacy
from pathlib import Path
from collections import Counter
import csv, re

print('spaCy version:', spacy.__version__)
nlp = spacy.load('fr_core_news_md')
print('Model:', nlp.meta['name'], nlp.meta['version'])
print('Pipeline:', nlp.pipe_names)

spaCy version: 3.8.18


Model: core_news_md 3.8.0
Pipeline: ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


### Output for one sentence (hand-in item 1)

In [3]:
sentence = "L'ADEME estime que les émissions de gaz à effet de serre de la France ont baissé de 5,8 % en 2023."
doc = nlp(sentence)
print(f"{'TOKEN':<14}{'LEMMA':<14}{'POS':<8}{'DEP':<12}HEAD")
for t in doc:
    print(f"{t.text:<14}{t.lemma_:<14}{t.pos_:<8}{t.dep_:<12}{t.head.text}")
print('\nEntities:', [(e.text, e.label_) for e in doc.ents])

TOKEN         LEMMA         POS     DEP         HEAD
L'            le            DET     det         ADEME
ADEME         ademe         NOUN    nsubj       estime
estime        estime        VERB    ROOT        estime
que           que           SCONJ   mark        baissé
les           le            DET     det         émissions
émissions     émission      NOUN    nsubj       baissé
de            de            ADP     case        gaz
gaz           gaz           NOUN    nmod        émissions
à             à             ADP     case        effet
effet         effet         NOUN    nmod        émissions
de            de            ADP     case        serre
serre         serre         NOUN    nmod        effet
de            de            ADP     case        France
la            le            DET     det         France
France        France        PROPN   nmod        serre
ont           avoir         AUX     aux:tense   baissé
baissé        baisser       VERB    ccomp       estime
de         

### French-specific check: elision and contractions
Discussion question 2 asks how each language handles contractions such as *l'homme*. Let's see how spaCy splits French elisions (*l', d', qu'*) and contractions (*du = de + le*, *aux = à + les*).

In [4]:
doc = nlp("L'énergie d'origine renouvelable qu'on produit aux Pays-Bas réduit l'usage du charbon.")
print([t.text for t in doc])
print([t.lemma_ for t in doc])

["L'", 'énergie', "d'", 'origine', 'renouvelable', "qu'", 'on', 'produit', 'aux', 'Pays', '-', 'Bas', 'réduit', "l'", 'usage', 'du', 'charbon', '.']
['le', 'énergie', 'de', 'origine', 'renouvelable', 'que', 'on', 'produire', 'à', 'Pays', '-', 'Bas', 'réduire', 'le', 'usage', 'de', 'charbon', '.']


## 2. Visualization with displaCy (hand-in item 2)

In [5]:
doc = nlp("La France a ratifié l'Accord de Paris lors de la COP21 en 2015.")
displacy.render(doc, style='dep', jupyter=True, options={'compact': True, 'distance': 110})

In [6]:
displacy.render(doc, style='ent', jupyter=True)

In [7]:
# Save the images as files too (useful for the report / GitHub)
Path('images').mkdir(exist_ok=True)
Path('images/dependency.svg').write_text(
    displacy.render(doc, style='dep', jupyter=False, options={'compact': True, 'distance': 110}), encoding='utf-8')
Path('images/entities.html').write_text(
    displacy.render(doc, style='ent', jupyter=False, page=True), encoding='utf-8')
print('Saved images/dependency.svg and images/entities.html')

Saved images/dependency.svg and images/entities.html


## 3. Loading and measuring the corpus (section 6.4)
The texts were collected with `collect_corpus.py --week1` (French Wikipedia API) and saved as UTF-8 files `corpus/text01.txt` … `corpus/text32.txt`. The corpus has grown since (see Week 2), so this notebook selects the Week 1 texts with the `added` column of `metadata.csv`.

In [8]:
# The corpus grows every week. This notebook uses the Week 1 texts only (column 'added' = week1).
with open('corpus/metadata.csv', encoding='utf-8') as f:
    meta = {row['file']: row for row in csv.DictReader(f)}
files = [Path('corpus') / name for name, row in meta.items() if row.get('added', 'week1') == 'week1']
texts = [f.read_text(encoding='utf-8') for f in files]

docs = list(nlp.pipe(texts))
tokens = [t.text.lower() for d in docs for t in d if t.is_alpha]

print('Texts:', len(texts))
print('Tokens:', len(tokens))
print('Unique words:', len(set(tokens)))
print(Counter(tokens).most_common(10))

Texts: 32
Tokens: 7446
Unique words: 2063
[('de', 510), ('la', 272), ('et', 205), ('des', 197), ('à', 193), ('les', 176), ('en', 163), ('le', 159), ('est', 119), ('par', 106)]


**Observation:** as the tutorial predicts, the ten most frequent words are mostly French stop words (*de, la, le, les, des, et, en, …*). They say almost nothing about the topic. Removing stop words and using lemmas shows the actual environmental vocabulary:

In [9]:
content = [t.lemma_.lower() for d in docs for t in d if t.is_alpha and not t.is_stop]
print('Top 15 content lemmas:', Counter(content).most_common(15))

Top 15 content lemmas: [('énergie', 76), ('émission', 54), ('carbone', 41), ('gaz', 38), ('serre', 25), ('eau', 23), ('climatique', 23), ('pollution', 21), ('mondial', 19), ('pays', 19), ('renouvelable', 19), ('produire', 16), ('source', 16), ('production', 16), ('réduire', 15)]


In [10]:
# Texts and tokens per category (the labels we will use for classification later)
per_cat = Counter()
tok_cat = Counter()
for f, d in zip(files, docs):
    cat = meta[f.name]['category']
    per_cat[cat] += 1
    tok_cat[cat] += sum(t.is_alpha for t in d)
for cat in per_cat:
    print(f"{cat:<25} texts={per_cat[cat]:<4} tokens={tok_cat[cat]}")

emissions                 texts=8    tokens=2049
energies_renouvelables    texts=8    tokens=1823
pollution                 texts=8    tokens=1841
politique_climatique      texts=8    tokens=1733


## 4. Looking for quality problems (section 6.5)

In [11]:
problems = []
for f, text in zip(files, texts):
    if re.search(r'<[^>]+>', text):
        problems.append((f.name, 'HTML tag'))
    if re.search(r'Ã[-¿]|â€', text):  # e.g. 'Ã©' instead of 'é' (plain 'Â' is valid: Moyen Âge)
        problems.append((f.name, 'mojibake (wrong encoding)'))
    lines = [l for l in text.splitlines() if l.strip()]
    if len(lines) != len(set(lines)):
        problems.append((f.name, 'repeated line'))
    if re.search(r'\(\s*\)|\[\d+\]|Modifier|Article détaillé', text):
        problems.append((f.name, 'Wikipedia leftover (empty brackets / references / edit links)'))
    n = len(text.split())
    if not 100 <= n <= 300:
        problems.append((f.name, f'length {n} words (outside 100–300)'))

# Duplicate texts
seen = {}
for f, text in zip(files, texts):
    if text in seen:
        problems.append((f.name, f'duplicate of {seen[text]}'))
    seen.setdefault(text, f.name)

# Sentences in another language: very few French stop words is suspicious
FR = {'le','la','les','de','des','du','et','est','en','un','une','que','qui','dans','pour','par','sur'}
for f, d in zip(files, docs):
    for s in d.sents:
        words = [t.text.lower() for t in s if t.is_alpha]
        if len(words) >= 8 and sum(w in FR for w in words) / len(words) < 0.05:
            problems.append((f.name, f'possibly non-French sentence: {s.text[:60]}…'))

# Accents check: special characters display correctly
accents = Counter(ch for t in texts for ch in t if ch in 'éèêàâçùûôîïëœÉÈÀÇ')
print('Accented characters found:', dict(accents.most_common()))
print(f'\n{len(problems)} problem(s) found:')
for p in problems:
    print(' -', *p)

Accented characters found: {'é': 1177, 'à': 195, 'è': 159, 'ê': 39, 'ç': 17, 'ï': 14, 'É': 14, 'î': 11, 'â': 9, 'û': 7, 'ù': 5, 'ô': 5, 'À': 2, 'œ': 1}

0 problem(s) found:


### Three sentences from the corpus: one thing right, one thing wrong (hand-in item 3)

In [12]:
# One sentence from three different categories
chosen = []
for cat in ['politique_climatique', 'energies_renouvelables', 'pollution']:
    f = next(f for f in files if meta[f.name]['category'] == cat)
    d = docs[files.index(f)]
    chosen.append((f.name, next(d.sents).text))

for i, (name, s) in enumerate(chosen, 1):
    print(f'Sentence {i} ({name}): {s}\n')

Sentence 1 (text25.txt): L'accord de Paris, ou accord de Paris sur le climat, est un traité international sur l'atténuation et l'adaptation au changement climatique, ainsi que sur leur financement suffisant.

Sentence 2 (text09.txt): Les énergies renouvelables (EnR) proviennent de sources d'énergie dont le renouvellement naturel est assez rapide pour qu'elles puissent être considérées comme inépuisables à l'échelle du temps humain.

Sentence 3 (text17.txt): La pollution de l'air (ou pollution atmosphérique) est une altération de la qualité de l'air pouvant être caractérisée par des mesures de polluants chimiques, biologiques ou physiques (appelés « aérocontaminants »).



In [13]:
for name, s in chosen:
    d = nlp(s)
    displacy.render(d, style='ent', jupyter=True)
    displacy.render(d, style='dep', jupyter=True, options={'compact': True, 'distance': 90})
    print('Entities:', [(e.text, e.label_) for e in d.ents])
    print('POS:', [(t.text, t.pos_) for t in d][:25])
    print('-' * 80)

Entities: [('Paris', 'LOC'), ('Paris', 'LOC')]
POS: [("L'", 'DET'), ('accord', 'NOUN'), ('de', 'ADP'), ('Paris', 'PROPN'), (',', 'PUNCT'), ('ou', 'CCONJ'), ('accord', 'NOUN'), ('de', 'ADP'), ('Paris', 'PROPN'), ('sur', 'ADP'), ('le', 'DET'), ('climat', 'NOUN'), (',', 'PUNCT'), ('est', 'AUX'), ('un', 'DET'), ('traité', 'NOUN'), ('international', 'ADJ'), ('sur', 'ADP'), ("l'", 'DET'), ('atténuation', 'NOUN'), ('et', 'CCONJ'), ("l'", 'DET'), ('adaptation', 'NOUN'), ('au', 'ADP'), ('changement', 'NOUN')]
--------------------------------------------------------------------------------


Entities: [('EnR', 'ORG')]
POS: [('Les', 'DET'), ('énergies', 'NOUN'), ('renouvelables', 'ADJ'), ('(', 'PUNCT'), ('EnR', 'NOUN'), (')', 'PUNCT'), ('proviennent', 'VERB'), ('de', 'ADP'), ('sources', 'NOUN'), ("d'", 'ADP'), ('énergie', 'NOUN'), ('dont', 'PRON'), ('le', 'DET'), ('renouvellement', 'NOUN'), ('naturel', 'ADJ'), ('est', 'AUX'), ('assez', 'ADV'), ('rapide', 'ADJ'), ('pour', 'ADP'), ("qu'", 'SCONJ'), ('elles', 'PRON'), ('puissent', 'VERB'), ('être', 'AUX'), ('considérées', 'VERB'), ('comme', 'ADP')]
--------------------------------------------------------------------------------


/usr/local/lib/python3.13/site-packages/spacy/displacy/__init__.py:214: UserWarning: [W006] No entities to visualize found in Doc object. If this is surprising to you, make sure the Doc was processed using a model that supports named entity recognition, and check the `doc.ents` property manually if necessary.
  warnings.warn(Warnings.W006)


Entities: []
POS: [('La', 'DET'), ('pollution', 'NOUN'), ('de', 'ADP'), ("l'", 'DET'), ('air', 'NOUN'), ('(', 'PUNCT'), ('ou', 'CCONJ'), ('pollution', 'NOUN'), ('atmosphérique', 'ADJ'), (')', 'PUNCT'), ('est', 'AUX'), ('une', 'DET'), ('altération', 'NOUN'), ('de', 'ADP'), ('la', 'DET'), ('qualité', 'NOUN'), ('de', 'ADP'), ("l'", 'DET'), ('air', 'NOUN'), ('pouvant', 'VERB'), ('être', 'AUX'), ('caractérisée', 'VERB'), ('par', 'ADP'), ('des', 'DET'), ('mesures', 'NOUN')]
--------------------------------------------------------------------------------


| # | Sentence | ✔ What the pipeline got right | ✘ What the pipeline got wrong |
|---|---|---|---|
| 1 | *L'accord de Paris, ou accord de Paris sur le climat, est un traité international…* (text25) | Elisions are split and lemmatized (*L'* → *le*, *l'atténuation*), and POS tags are correct (*accord* NOUN, *Paris* PROPN, *traité* NOUN, *international* ADJ). | NER does not see *accord de Paris* as one entity (a treaty, MISC). It only tags *Paris* as a place (LOC), twice. |
| 2 | *Les énergies renouvelables (EnR) proviennent de sources d'énergie…* (text09) | *d'* and *qu'* are split from the next word. The verbs are lemmatized correctly (*proviennent* → *provenir*, *puissent* → *pouvoir*), and *EnR* is attached to *énergies* as an apposition. | The abbreviation *EnR* is labelled as an organisation (ORG). The pronoun *elles* gets the wrong lemma *lui*. *inépuisables* is wrongly attached to *comme* as `fixed`. |
| 3 | *La pollution de l'air (ou pollution atmosphérique) est une altération de la qualité de l'air…* (text17) | The copula structure is correct (*altération* is ROOT, *est* is `cop`). The passive *pouvant être caractérisée par des mesures* is parsed correctly (`aux:pass`, `obl:agent`). It correctly finds no named entities. | The brackets are attached inconsistently: the second *(* goes to *polluants* but its closing *)* goes to *altération*. *(ou pollution atmosphérique)* is parsed as a coordination (`conj`) instead of an apposition (it is a synonym). |

## 5. Comparing the groups — our row (section 6.6)

In [14]:
row = {
    'Language': 'French',
    'Source': 'Wikipédia (fr), article introductions',
    'Licence': 'CC BY-SA 4.0',
    'Texts': len(texts),
    'Tokens': len(tokens),
    'Unique words': len(set(tokens)),
    'Problems found': len(problems),
}
for k, v in row.items():
    print(f'{k:<15}{v}')

Language       French
Source         Wikipédia (fr), article introductions
Licence        CC BY-SA 4.0
Texts          32
Tokens         7446
Unique words   2063
Problems found 0


**Notes for the discussion**
- **Accents:** all accented characters are preserved because every file is read and written in UTF-8.
- **Elision / contractions:** see section 1. Elided forms (*l', d', qu'*) are split correctly and lemmatized (*l'* → *le*, *qu'* → *que*). Contractions are **not** split: *du* stays one token with lemma *de*, and *aux* one token with lemma *à*, so the article (*le / les*) is lost.
- **Compounds:** hyphenated names are split, e.g. *Pays-Bas* → `Pays`, `-`, `Bas`.
- **Quality check false alarm:** a simple mojibake check flagged *Moyen Âge* (text15) because `Â` is also a typical encoding error. The check now only looks for real mojibake patterns like `Ã©`.
- **Acronyms:** in the section 1 sentence, NER correctly tags *ADEME* as ORG, but the POS tagger calls it a common noun (NOUN, not PROPN) with lemma *ademe*. In sentence 2, the abbreviation *EnR* is wrongly tagged ORG. Domain acronyms (ADEME, GIEC, EnR, COP21) are hard for the general French model.
- **Before a real project:** add more texts (target ~100, about 25 per category), balance the categories, and consider adding official sources (ADEME, vie-publique.fr, EU) after checking their licences.

## 6. Sources and licences (hand-in item 5)

In [15]:
print(f"{'File':<12}{'Category':<24}{'Title':<55}Licence")
for f in files:
    m = meta[f.name]
    print(f"{f.name:<12}{m['category']:<24}{m['title'][:53]:<55}{m['licence']}")
print('\nAll texts: Wikipédia (fr), CC BY-SA 4.0 — https://creativecommons.org/licenses/by-sa/4.0/')
print('Full URLs and dates: corpus/metadata.csv')

File        Category                Title                                                  Licence
text01.txt  emissions               Dioxyde de carbone                                     CC BY-SA 4.0
text02.txt  emissions               Méthane                                                CC BY-SA 4.0
text03.txt  emissions               Empreinte carbone                                      CC BY-SA 4.0
text04.txt  emissions               Bilan carbone                                          CC BY-SA 4.0
text05.txt  emissions               Effet de serre                                         CC BY-SA 4.0
text06.txt  emissions               Neutralité carbone                                     CC BY-SA 4.0
text07.txt  emissions               Protoxyde d'azote                                      CC BY-SA 4.0
text08.txt  emissions               Déforestation                                          CC BY-SA 4.0
text09.txt  energies_renouvelables  Énergie renouvelable             